# Explore WordNet with NLTK

This notebook helps you inspect **WordNet** entries for any English word (with a focus on nouns).

## What is WordNet?
WordNet is a large lexical database of English where words are grouped into sets of cognitive synonyms called **synsets**.

## What is a synset?
A **synset** is a specific sense of a word (for example, one sense of `bank` means a financial institution, another means land by a river).

## What this notebook displays
For each synset found for a word, this notebook shows:
- Synset name
- Part of speech
- Definition (gloss)
- Example sentences
- Lemmas (synonyms)
- Hypernyms and hyponyms
- Holonyms and meronyms (member/part/substance)
- Attributes, entailments, and `similar_tos` (when available)

It also includes optional advanced helpers to inspect hypernym paths and compare two words side by side.

In [1]:
# Setup cell: ensure NLTK and WordNet resources are available.
# This cell is robust: it installs/downloads only when needed.

import sys
import subprocess
import importlib

def _ensure_package(package_name):
    """Install a Python package with pip if it is not already importable."""
    try:
        importlib.import_module(package_name)
        print(f"Package '{package_name}' is already installed.")
    except ImportError:
        print(f"Package '{package_name}' not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        print(f"Installed '{package_name}'.")

# Ensure NLTK itself is installed.
_ensure_package("nltk")

import nltk
from nltk.corpus import wordnet as wn

def _ensure_nltk_resource(resource_path, download_name):
    """Download an NLTK corpus/model if missing."""
    try:
        nltk.data.find(resource_path)
        print(f"NLTK resource '{download_name}' is available.")
    except LookupError:
        print(f"NLTK resource '{download_name}' missing. Downloading...")
        nltk.download(download_name)

# WordNet corpus and multilingual index used by many WordNet installs.
_ensure_nltk_resource("corpora/wordnet", "wordnet")
_ensure_nltk_resource("corpora/omw-1.4", "omw-1.4")

print("Setup complete. Ready to inspect WordNet.")

Package 'nltk' is already installed.
NLTK resource 'wordnet' missing. Downloading...
NLTK resource 'omw-1.4' missing. Downloading...
Setup complete. Ready to inspect WordNet.


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\xiaoy\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\xiaoy\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
# Core inspection utilities.
# Main entry point: inspect_wordnet(word, pos=None, noun_only_words=False, exact_match_only=False)

from typing import List, Optional, Tuple

# Human-friendly labels and WordNet POS mappings.
POS_MAP = {
    None: None,
    "noun": wn.NOUN,
    "verb": wn.VERB,
    "adjective": wn.ADJ,
    "adverb": wn.ADV,
}

POS_LABEL = {
    "n": "noun",
    "v": "verb",
    "a": "adjective",
    "s": "adjective satellite",
    "r": "adverb",
}

def _clean_list(values: List[str], empty_text: str = "(none)") -> str:
    """Return a readable comma-separated string, even for empty lists."""
    if not values:
        return empty_text
    return ", ".join(values)

def _synset_items(synsets) -> List[str]:
    """Format related synsets as 'name - definition'."""
    items = []
    for s in synsets:
        try:
            items.append(f"{s.name()} - {s.definition()}")
        except Exception:
            # Fallback if any individual synset is malformed.
            items.append(str(s))
    return items

def _print_section(title: str, values: List[str]):
    """Consistent section rendering for notebook readability."""
    print(f"  {title}:")
    if not values:
        print("    (none)")
        return
    for item in values:
        print(f"    - {item}")

def _normalize_lookup(text: str) -> str:
    """Normalize user input and lemma names for exact matching."""
    return text.replace("_", " ").strip().lower()

def _synset_head_word(syn) -> str:
    """Return the canonical word form used by the synset name."""
    return syn.name().split(".", 1)[0].replace("_", " ")

def _filter_synsets(
    synsets,
    word: str,
    noun_only_words: bool,
    exact_match_only: bool,
) -> List[Tuple[object, List[str]]]:
    """Filter synsets by POS and canonical word constraints."""
    if not noun_only_words and not exact_match_only:
        return [
            (syn, [lemma.name().replace("_", " ") for lemma in syn.lemmas()])
            for syn in synsets
        ]

    normalized_word = _normalize_lookup(word)
    filtered_synsets = []

    for syn in synsets:
        if noun_only_words and syn.pos() != wn.NOUN:
            continue

        if exact_match_only and _normalize_lookup(_synset_head_word(syn)) != normalized_word:
            continue

        matched_lemmas = []
        for lemma in syn.lemmas():
            lemma_name = lemma.name()
            if exact_match_only and _normalize_lookup(lemma_name) != normalized_word:
                continue
            matched_lemmas.append(lemma_name.replace("_", " "))

        if matched_lemmas:
            filtered_synsets.append((syn, matched_lemmas))

    return filtered_synsets

def collect_wordnet_definitions(
    word: str,
    pos: Optional[str] = None,
    noun_only_words: bool = True,
    exact_match_only: bool = True,
) -> List[str]:
    """Return all Definition texts from the filtered inspect_wordnet results."""
    if not isinstance(word, str) or not word.strip():
        print("Please provide a non-empty word string.")
        return []

    normalized_word = word.strip()
    normalized_pos = pos.lower().strip() if isinstance(pos, str) else None

    if normalized_pos not in POS_MAP:
        valid = ["noun", "verb", "adjective", "adverb", "None"]
        print(f"Invalid pos='{pos}'. Valid options: {', '.join(valid)}")
        return []

    wn_pos = POS_MAP[normalized_pos]
    raw_synsets = wn.synsets(normalized_word, pos=wn_pos) if wn_pos else wn.synsets(normalized_word)
    synsets = _filter_synsets(raw_synsets, normalized_word, noun_only_words, exact_match_only)
    return [syn.definition() if syn.definition() else "(none)" for syn, _ in synsets]

def inspect_wordnet(
    word: str,
    pos: Optional[str] = None,
    noun_only_words: bool = False,
    exact_match_only: bool = False,
):
    """
    Inspect WordNet synsets for a given word.

    Parameters
    ----------
    word : str
        Input word to inspect.
    pos : str | None
        Optional POS filter: noun, verb, adjective, adverb, or None.
    noun_only_words : bool
        If True, keep only noun synsets in the returned results.
    exact_match_only : bool
        If True, keep only synsets whose canonical word exactly matches the input word.
    """
    if not isinstance(word, str) or not word.strip():
        print("Please provide a non-empty word string.")
        return

    normalized_word = word.strip()
    normalized_pos = pos.lower().strip() if isinstance(pos, str) else None

    if normalized_pos not in POS_MAP:
        valid = ["noun", "verb", "adjective", "adverb", "None"]
        print(f"Invalid pos='{pos}'. Valid options: {', '.join(valid)}")
        return

    wn_pos = POS_MAP[normalized_pos]

    # Retrieve synsets with optional POS filtering.
    raw_synsets = wn.synsets(normalized_word, pos=wn_pos) if wn_pos else wn.synsets(normalized_word)
    synsets = _filter_synsets(raw_synsets, normalized_word, noun_only_words, exact_match_only)

    print("=" * 90)
    print(f"WordNet inspection for word: '{normalized_word}'")
    print(f"POS filter: {normalized_pos if normalized_pos else 'None (all parts of speech)'}")
    print(f"Noun-only synsets: {noun_only_words}")
    print(f"Exact lemma match: {exact_match_only}")
    print(f"Synsets found: {len(synsets)}")
    print("=" * 90)

    # Robust handling when no synsets exist after filtering.
    if not synsets:
        print("No synsets found for this word with the selected filters.")
        return

    for idx, (syn, lemmas) in enumerate(synsets, start=1):
        print(f"\n[{idx}] {syn.name()}")
        print("-" * 90)

        # Basic synset fields.
        pos_label = POS_LABEL.get(syn.pos(), syn.pos())
        print(f"  Synset name : {syn.name()}")
        print(f"  POS         : {pos_label}")
        print(f"  Definition  : {syn.definition() if syn.definition() else '(none)'}")

        examples = syn.examples() or []
        _print_section("Example sentences", examples)

        _print_section("Lemma names (synonyms)", lemmas)

        # Relation sections required by the notebook specification.
        _print_section("Hypernyms", _synset_items(syn.hypernyms()))
        _print_section("Hyponyms", _synset_items(syn.hyponyms()))

        _print_section("Member holonyms", _synset_items(syn.member_holonyms()))
        _print_section("Part holonyms", _synset_items(syn.part_holonyms()))
        _print_section("Substance holonyms", _synset_items(syn.substance_holonyms()))

        _print_section("Member meronyms", _synset_items(syn.member_meronyms()))
        _print_section("Part meronyms", _synset_items(syn.part_meronyms()))
        _print_section("Substance meronyms", _synset_items(syn.substance_meronyms()))

        _print_section("Attributes", _synset_items(syn.attributes()))
        _print_section("Entailments", _synset_items(syn.entailments()))

        # similar_tos is mostly meaningful for adjective-like synsets.
        similar_tos = syn.similar_tos() if hasattr(syn, "similar_tos") else []
        _print_section("Similar_tos", _synset_items(similar_tos))

        print("-" * 90)

    print("\nInspection complete.")


## Optional Advanced Helpers

This section includes:
1. A helper to print hypernym path tree(s) for a selected synset.
2. A helper to compare two words by listing their synsets side by side.

In [3]:
# Advanced helper functions.

import textwrap

def print_hypernym_paths_for_word(word: str, pos: Optional[str] = None, synset_index: int = 0):
    """
    Print all hypernym paths for one selected synset of a word.

    Parameters
    ----------
    word : str
        Target word.
    pos : str | None
        Optional POS filter: noun, verb, adjective, adverb, or None.
    synset_index : int
        Zero-based index of the synset to inspect.
    """
    normalized_pos = pos.lower().strip() if isinstance(pos, str) else None
    if normalized_pos not in POS_MAP:
        print(f"Invalid pos='{pos}'.")
        return

    wn_pos = POS_MAP[normalized_pos]
    synsets = wn.synsets(word, pos=wn_pos) if wn_pos else wn.synsets(word)

    if not synsets:
        print(f"No synsets found for '{word}'.")
        return

    if synset_index < 0 or synset_index >= len(synsets):
        print(f"synset_index out of range. Choose 0 to {len(synsets)-1}.")
        return

    target = synsets[synset_index]
    paths = target.hypernym_paths() or []

    print("=" * 90)
    print(f"Hypernym paths for: {target.name()} — {target.definition()}")
    print(f"Total paths: {len(paths)}")
    print("=" * 90)

    if not paths:
        print("No hypernym paths available for this synset.")
        return

    for i, path in enumerate(paths, start=1):
        print(f"\nPath {i}:")
        for depth, node in enumerate(path):
            indent = "  " * depth
            print(f"{indent}- {node.name()} ({POS_LABEL.get(node.pos(), node.pos())})")

def compare_words_synsets(word1: str, word2: str, pos: Optional[str] = None, max_synsets: int = 8):
    """
    Compare synsets of two words side by side in plain text.
    """
    normalized_pos = pos.lower().strip() if isinstance(pos, str) else None
    if normalized_pos not in POS_MAP:
        print(f"Invalid pos='{pos}'.")
        return

    wn_pos = POS_MAP[normalized_pos]
    synsets1 = wn.synsets(word1, pos=wn_pos) if wn_pos else wn.synsets(word1)
    synsets2 = wn.synsets(word2, pos=wn_pos) if wn_pos else wn.synsets(word2)

    print("=" * 120)
    print(f"Synset comparison: '{word1}' vs '{word2}' | POS: {normalized_pos if normalized_pos else 'all'}")
    print(f"Counts: {len(synsets1)} vs {len(synsets2)}")
    print("=" * 120)

    if not synsets1 and not synsets2:
        print("No synsets found for either word.")
        return

    width = 58
    rows = max(min(len(synsets1), max_synsets), min(len(synsets2), max_synsets), 1)

    def _syn_line(s):
        if s is None:
            return "(none)"
        return f"{s.name()} — {s.definition()}"

    header_left = f"{word1} synsets"
    header_right = f"{word2} synsets"
    print(f"{header_left:<{width}} | {header_right:<{width}}")
    print("-" * width + "-+-" + "-" * width)

    for i in range(rows):
        s1 = synsets1[i] if i < len(synsets1) and i < max_synsets else None
        s2 = synsets2[i] if i < len(synsets2) and i < max_synsets else None

        left = textwrap.shorten(_syn_line(s1), width=width, placeholder="...")
        right = textwrap.shorten(_syn_line(s2), width=width, placeholder="...")
        print(f"{left:<{width}} | {right:<{width}}")

    if len(synsets1) > max_synsets or len(synsets2) > max_synsets:
        print("\n(Comparison truncated by max_synsets.)")

## Demo Cells

In [4]:
# Demo: director
inspect_wordnet("director")

WordNet inspection for word: 'director'
POS filter: None (all parts of speech)
Noun-only synsets: False
Exact lemma match: False
Synsets found: 5

[1] director.n.01
------------------------------------------------------------------------------------------
  Synset name : director.n.01
  POS         : noun
  Definition  : someone who controls resources and expenditures
  Example sentences:
    (none)
  Lemma names (synonyms):
    - director
    - manager
    - managing director
  Hypernyms:
    - administrator.n.01 - someone who administers a business
  Hyponyms:
    - bank_manager.n.01 - manager of a branch office of a bank
    - district_manager.n.01 - a manager who supervises the sales activity for a district
    - manageress.n.01 - a woman manager
  Member holonyms:
    (none)
  Part holonyms:
    (none)
  Substance holonyms:
    (none)
  Member meronyms:
    (none)
  Part meronyms:
    (none)
  Substance meronyms:
    (none)
  Attributes:
    (none)
  Entailments:
    (none)
  Simi

In [5]:
# Demo: career
inspect_wordnet("career")

WordNet inspection for word: 'career'
POS filter: None (all parts of speech)
Noun-only synsets: False
Exact lemma match: False
Synsets found: 3

[1] career.n.01
------------------------------------------------------------------------------------------
  Synset name : career.n.01
  POS         : noun
  Definition  : the particular occupation for which you are trained
  Example sentences:
    (none)
  Lemma names (synonyms):
    - career
    - calling
    - vocation
  Hypernyms:
    - occupation.n.01 - the principal activity in your life that you do to earn money
  Hyponyms:
    - lifework.n.01 - the principal work of your career
    - walk_of_life.n.01 - careers in general
    - business_life.n.01 - a career in industrial or commercial or professional activities
    - specialization.n.02 - the special line of work you have adopted as your career
  Member holonyms:
    (none)
  Part holonyms:
    (none)
  Substance holonyms:
    (none)
  Member meronyms:
    (none)
  Part meronyms:
    (

In [6]:
# Demo: bank
inspect_wordnet("us")

WordNet inspection for word: 'bank'
POS filter: None (all parts of speech)
Noun-only synsets: False
Exact lemma match: False
Synsets found: 18

[1] bank.n.01
------------------------------------------------------------------------------------------
  Synset name : bank.n.01
  POS         : noun
  Definition  : sloping land (especially the slope beside a body of water)
  Example sentences:
    - they pulled the canoe up on the bank
    - he sat on the bank of the river and watched the currents
  Lemma names (synonyms):
    - bank
  Hypernyms:
    - slope.n.01 - an elevated geological formation
  Hyponyms:
    - riverbank.n.01 - the bank of a river
    - waterside.n.01 - land bordering a body of water
  Member holonyms:
    (none)
  Part holonyms:
    (none)
  Substance holonyms:
    (none)
  Member meronyms:
    (none)
  Part meronyms:
    (none)
  Substance meronyms:
    (none)
  Attributes:
    (none)
  Entailments:
    (none)
  Similar_tos:
    (none)
--------------------------------

In [14]:
# Demo: apple
inspect_wordnet("madonna",noun_only_words=True,exact_match_only=True)

WordNet inspection for word: 'us'
POS filter: None (all parts of speech)
Noun-only synsets: True
Exact lemma match: True
Synsets found: 0
No synsets found for this word with the selected filters.


In [13]:
collect_wordnet_definitions("us")

[]

In [9]:
# Noun-only inspection example (requirement 8).
inspect_wordnet("bank", pos="noun")

WordNet inspection for word: 'bank'
POS filter: noun
Noun-only synsets: False
Exact lemma match: False
Synsets found: 10

[1] bank.n.01
------------------------------------------------------------------------------------------
  Synset name : bank.n.01
  POS         : noun
  Definition  : sloping land (especially the slope beside a body of water)
  Example sentences:
    - they pulled the canoe up on the bank
    - he sat on the bank of the river and watched the currents
  Lemma names (synonyms):
    - bank
  Hypernyms:
    - slope.n.01 - an elevated geological formation
  Hyponyms:
    - riverbank.n.01 - the bank of a river
    - waterside.n.01 - land bordering a body of water
  Member holonyms:
    (none)
  Part holonyms:
    (none)
  Substance holonyms:
    (none)
  Member meronyms:
    (none)
  Part meronyms:
    (none)
  Substance meronyms:
    (none)
  Attributes:
    (none)
  Entailments:
    (none)
  Similar_tos:
    (none)
------------------------------------------------------

In [10]:
# Advanced demo 1: hypernym path tree for the first noun sense of 'apple'.
print_hypernym_paths_for_word("apple", pos="noun", synset_index=0)

Hypernym paths for: apple.n.01 — fruit with red or yellow or green skin and sweet to tart crisp whitish flesh
Total paths: 3

Path 1:
- entity.n.01 (noun)
  - physical_entity.n.01 (noun)
    - matter.n.03 (noun)
      - solid.n.01 (noun)
        - food.n.02 (noun)
          - produce.n.01 (noun)
            - edible_fruit.n.01 (noun)
              - apple.n.01 (noun)

Path 2:
- entity.n.01 (noun)
  - physical_entity.n.01 (noun)
    - object.n.01 (noun)
      - whole.n.02 (noun)
        - natural_object.n.01 (noun)
          - plant_part.n.01 (noun)
            - plant_organ.n.01 (noun)
              - reproductive_structure.n.01 (noun)
                - fruit.n.01 (noun)
                  - edible_fruit.n.01 (noun)
                    - apple.n.01 (noun)

Path 3:
- entity.n.01 (noun)
  - physical_entity.n.01 (noun)
    - object.n.01 (noun)
      - whole.n.02 (noun)
        - natural_object.n.01 (noun)
          - plant_part.n.01 (noun)
            - plant_organ.n.01 (noun)
            

In [11]:
# Advanced demo 2: side-by-side comparison.
compare_words_synsets("bank", "career", pos=None, max_synsets=6)

Synset comparison: 'bank' vs 'career' | POS: all
Counts: 18 vs 3
bank synsets                                               | career synsets                                            
-----------------------------------------------------------+-----------------------------------------------------------
bank.n.01 — sloping land (especially the slope beside a... | career.n.01 — the particular occupation for which you...  
depository_financial_institution.n.01 — a financial...     | career.n.02 — the general progression of your working...  
bank.n.03 — a long ridge or pile                           | career.v.01 — move headlong at high speed                 
bank.n.04 — an arrangement of similar objects in a row...  | (none)                                                    
bank.n.05 — a supply or stock held in reserve for...       | (none)                                                    
bank.n.06 — the funds held by a gambling house or the...   | (none)                            